# 0.25 — The raw news feed: anatomy, the **replay-date artifact**, and the fix

The `data/raw/raw_news_{year}.csv.xz` files are **not** a list of articles — they are a capture of
Bloomberg's machine-readable news feed. **Each row is a feed *message* about a story**, timestamped
at transmission (`CaptureTime`). One story emits many messages over its life, and one message class
(`UPDATE_ATTRIBUTE`) can re-transmit a story **years after publication**.

The legacy pipeline (per-year `terms_{year}.parquet` and everything built on top) treated
`CaptureTime` as the publication date. This notebook shows why that is wrong for a large,
bursty slice of the data — it **manufactures fake news spikes**, the exact thing our emergence
detectors look for — and states the fix (implemented in `scripts/preprocess_news.py`, applied in 0.26).

Case study: the giant "hacking" spike of **June 2013** found in the ETF-theme cyber timeline (0.27).

In [1]:
import lzma, re
from pathlib import Path
import pandas as pd
import polars as pl
_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW = _ROOT / "data" / "raw"
WIRES = ["BN", "BFW", "BBO"]                       # Bloomberg-authored English wires (same as terms builder)

def load_raw(yr, columns=None, flt=None, head=None):
    """One year of the feed capture, optionally filtered/limited inside the lazy scan."""
    with lzma.open(RAW / f"raw_news_{yr}.csv.xz", "rb") as f:
        q = pl.scan_csv(f, infer_schema_length=10_000)
        if columns: q = q.select(columns)
        if flt is not None: q = q.filter(flt)
        if head: q = q.head(head)
        return q.collect()

# schema + a few example rows (tag columns truncated for display)
peek = load_raw(2013, head=1000)
print("columns:", peek.columns, "\n")
show = peek.head(4).with_columns([pl.col(c).cast(pl.Utf8).str.slice(0, 38) for c in
                                  ["DerivedTickersId", "AssignedTopicsId", "DerivedTopicsId"]])
print(show.select(["CaptureTime","Event","Version","WireName","Headline","DerivedTickersId","DerivedTopicsId"])
          .to_pandas().to_string(index=False))

columns: ['LanguageString', 'DerivedTickersId', 'AssignedTickersId', 'CaptureTime', 'Headline', 'AssignedTopicsId', 'DerivedTopicsId', 'Event', 'Version', 'WireName'] 

                  CaptureTime       Event  Version WireName                                                         Headline                       DerivedTickersId                        DerivedTopicsId
2013-01-01T00:00:00.604+00:00 ADD_1STPASS ORIGINAL       BN      Latin America Currency Update as of 7:00 p.m. New York Time %UYU;%PYG;%COP;%CLP;%PEN;%ARS;%MXN;%BR FRX;MARKETS;BIZNEWS;BUSINESS;FINNEWS;C
2013-01-01T00:00:00.825+00:00   ADD_STORY ORIGINAL       BN      Latin America Currency Update as of 7:00 p.m. New York Time %ARS;%BRL;%CLP;%COP;%MXN;%PEN;%PYG;%UY BGOVBILLGO;BIZNEWS;BONDWIRES;BUSINESS;
2013-01-01T00:00:01.008+00:00 ADD_1STPASS     None       BN Italy Debt Rallies in Euro-Area Bonds’ Best Year as Crisis Eases             1174Z@PL;2103Z@IM;2539Z@GR ANA;ANABON;AUSTRIA;AVB;BB;BBB;BIZNEWS;
2013-01-01T00:00:01

## 1 · What the columns mean

| column | meaning |
|---|---|
| `CaptureTime` | when this **message** was transmitted on the feed — *not* necessarily when the story was published (the crux of this notebook) |
| `Headline` | headline text of the story record at message time |
| `Event` | **why the message was sent**: `ADD_1STPASS` = first publication of the headline (the instant flash, often before the article body exists) · `ADD_STORY` = the full story record added to the wire · `UPDATE_ATTRIBUTE` = the story's *tags* were edited and the whole record is re-transmitted — seconds **or years** after publication |
| `Version` | revision state of the story *text*: `ORIGINAL` / `UPDATE` / `CORRECTION` (orthogonal to `Event`) |
| `WireName` | wire code: `BN` Bloomberg News, `BFW` Bloomberg First Word, `BBO` Bloomberg Opinion; the dump also carries ~220 third-party wires we exclude |
| `DerivedTickersId` / `AssignedTickersId` | tickers tagged on the story (machine-derived vs editor-assigned; `%XXX` = currencies) |
| `DerivedTopicsId` / `AssignedTopicsId` | topic codes (`BIZNEWS`, `FRX`, …) — **these are the "attributes" whose edits trigger `UPDATE_ATTRIBUTE`** |
| `LanguageString` | headline language |

**Publication time = the timestamp of the `ADD_*` messages.** `UPDATE_ATTRIBUTE` timestamps are
tag-edit times.

In [2]:
# full-year 2013 message mix (Bloomberg wires only) — loaded once, reused below
d13 = load_raw(2013, ["Headline", "CaptureTime", "Event", "Version", "WireName"],
               pl.col("WireName").is_in(WIRES) & pl.col("Headline").is_not_null()
              ).with_columns(pl.col("CaptureTime").str.to_datetime(time_zone="UTC", strict=False))
print(f"{len(d13):,} feed messages · {d13['Headline'].n_unique():,} distinct headlines\n")
mix = d13.group_by(["Event", "Version"]).len().sort("len", descending=True)
print(mix.head(8).to_pandas().to_string(index=False))
print("\n=> 72% of messages are UPDATE_ATTRIBUTE re-transmissions, not publications")

16,125,256 feed messages · 3,227,695 distinct headlines

           Event    Version     len
UPDATE_ATTRIBUTE   ORIGINAL 7518554
UPDATE_ATTRIBUTE     UPDATE 4058459
       ADD_STORY   ORIGINAL 2142130
     ADD_1STPASS   ORIGINAL 1795519
     ADD_1STPASS     UPDATE  328940
       ADD_STORY     UPDATE  154901
UPDATE_ATTRIBUTE CORRECTION   68683
     ADD_1STPASS       None   49239

=> 72% of messages are UPDATE_ATTRIBUTE re-transmissions, not publications


## 2 · The problem — replayed history wearing today's date

If a story's tags are edited long after publication, its old headline re-enters the capture with a
**fresh `CaptureTime`**. Downstream we keep `drop_duplicates("Headline")` — but only *per year-file*,
so a replay of a story published **before** the corpus window survives with the wrong date.

The result: a bulk re-tagging day looks like a **one-day news burst about old events** — precisely
the false positive an emergence detector must not swallow.

In [3]:
# case study: monthly "hack*" headlines in 2013 (dedup per current pipeline convention: within-year first)
HACK = r"(?i)\bhack(ed|ing|ers?)\b"
h13 = (d13.filter(pl.col("Headline").str.contains(HACK))
          .sort("CaptureTime").unique(subset="Headline", keep="first"))
monthly = h13.group_by(pl.col("CaptureTime").dt.strftime("%Y-%m").alias("month")).len().sort("month")
print("monthly distinct hack* headlines, 2013 (as currently dated):")
print(monthly.to_pandas().to_string(index=False))
jun = h13.filter(pl.col("CaptureTime").dt.month() == 6)
byday = jun.group_by(pl.col("CaptureTime").dt.date().alias("day")).len().sort("len", descending=True)
print(f"\nJune by day — top 3 of {len(byday)}:")
print(byday.head(3).to_pandas().to_string(index=False))
print("\nsample headlines from the 2013-06-23 batch (recognizably 2011-12 stories):")
for h in jun.filter(pl.col("CaptureTime").dt.day() == 23)["Headline"].head(6).to_list():
    print("  ", h[:95])

monthly distinct hack* headlines, 2013 (as currently dated):
  month  len
2013-01   46
2013-02   83
2013-03  109
2013-04   62
2013-05   42
2013-06 1365
2013-07   66
2013-08   36
2013-09   34
2013-10   48
2013-11   89
2013-12   29

June by day — top 3 of 15:
       day  len
2013-06-23 1312
2013-06-24    8
2013-06-05    7

sample headlines from the 2013-06-23 batch (recognizably 2011-12 stories):
   Fox News Apologizes for Hacked Twitter Claims Obama Killed
   *YATES SAYS HE RECEIVED ASSURANCES FROM WALLIS ON PHONE-HACKING
   Clegg Says Hacking Scandal Shows Establishment Became ‘Too Cosy’
   *MURDOCH SAYS HE WAS MORE WORRIED ABOUT HACKING SCANDAL :NWSA US
   U.K. Police Widen Phone-Hacking Probe to Add Money Laundering
   *LAWMAKERS TO RELEASE PHONE-HACKING REPORT MAY 1        :NWSA US


In [4]:
# the proof: the same headline text exists in the 2011/2012 captures with its ORIGINAL timestamp
HACK = r"(?i)\bhack(ed|ing|ers?)\b"                    # (re)derived from d13 so this cell runs standalone
h13 = (d13.filter(pl.col("Headline").str.contains(HACK))
          .sort("CaptureTime").unique(subset="Headline", keep="first"))
jun = h13.filter(pl.col("CaptureTime").dt.month() == 6)
batch = jun.filter(pl.col("CaptureTime").dt.day() == 23)["Headline"].unique().to_list()
print(f"unique hack* headlines captured 2013-06-23: {len(batch)}")
found = {}
for yr in (2011, 2012):
    m = load_raw(yr, ["Headline", "CaptureTime"], pl.col("Headline").is_in(batch)
        ).with_columns(pl.col("CaptureTime").str.to_datetime(time_zone="UTC", strict=False))
    found[yr] = m.sort("CaptureTime").unique(subset="Headline", keep="first")
    print(f"  found verbatim in raw_news_{yr}: {len(found[yr]):,}")
both = pl.concat(list(found.values())).sort("CaptureTime").unique(subset="Headline", keep="first")
print(f"total with an earlier twin: {len(both):,} / {len(batch)} = {len(both)/len(batch):.0%}\n")
print("their TRUE capture months (the scandal's real news cycle — July 2011 = Milly Dowler month):")
print(both.group_by(pl.col("CaptureTime").dt.strftime("%Y-%m").alias("month")).len()
          .sort("month").to_pandas().to_string(index=False))
ex = both.sort("CaptureTime").head(2)
for r in ex.iter_rows():
    print(f"\n  original {r[1]:%Y-%m-%d %H:%M}  ->  replayed 2013-06-23   {r[0][:70]}")

unique hack* headlines captured 2013-06-23: 1312
  found verbatim in raw_news_2011: 790
  found verbatim in raw_news_2012: 558
total with an earlier twin: 1,261 / 1312 = 96%

their TRUE capture months (the scandal's real news cycle — July 2011 = Milly Dowler month):
  month  len
2011-01   17
2011-02   13
2011-03   15
2011-04   27
2011-05   18
2011-06   18
2011-07  371
2011-08   83
2011-09   88
2011-10   39
2011-11   56
2011-12   45
2012-01   32
2012-02   89
2012-03   25
2012-04   83
2012-05   61
2012-06   31
2012-07   55
2012-08   29
2012-09   36
2012-10   16
2012-11    6
2012-12    8

  original 2011-01-05 17:47  ->  replayed 2013-06-23   *NEWS OF THE WORLD SUSPENDS ASSISTANT EDITOR OVER PHONE HACKING

  original 2011-01-05 17:50  ->  replayed 2013-06-23   News of the World Suspends Assistant Editor Over Phone Hacking


In [5]:
# generality: replay/bulk-retag days are ROUTINE, June 23 just happened to hit our keyword
first = d13.sort("CaptureTime").group_by("Headline", maintain_order=True).first()
ev = first.group_by("Event").len().sort("len", descending=True)
print("event type of each headline's FIRST occurrence within 2013:")
tot = len(first)
for e, n in ev.iter_rows(): print(f"  {e:18} {n:9,}  ({n/tot:.1%})")
uo = first.filter((pl.col("Event") == "UPDATE_ATTRIBUTE") & (pl.col("Version") == "ORIGINAL"))
top = uo.group_by(pl.col("CaptureTime").dt.date().alias("day")).len().sort("len", descending=True)
print(f"\nheadlines first seen as (UPDATE_ATTRIBUTE, ORIGINAL): {len(uo):,}"
      f" · median/day {top['len'].median():.0f} · top bulk days:")
print(top.head(5).to_pandas().to_string(index=False))
print("\n=> 35% of 'new' headlines enter the capture as re-emissions; bulk days reach 27k-63k")

event type of each headline's FIRST occurrence within 2013:
  ADD_1STPASS        1,987,715  (61.6%)
  UPDATE_ATTRIBUTE   1,138,193  (35.3%)
  ADD_STORY            101,787  (3.2%)

headlines first seen as (UPDATE_ATTRIBUTE, ORIGINAL): 1,016,725 · median/day 824 · top bulk days:
       day   len
2013-12-13 63515
2013-03-26 50602
2013-03-27 48299
2013-12-09 28160
2013-03-12 27933

=> 35% of 'new' headlines enter the capture as re-emissions; bulk days reach 27k-63k


## 3 · The solution

**Date every headline by its earliest occurrence across the *entire* capture (2010–2025), and let the
feed certify it.** Concretely, build once a global index `headline -> (first_capture, first_event)` and
use it wherever the corpus is built:

1. **Global first-occurrence dating** — `date := min(CaptureTime)` over all years, not within one
   year-file. Purely mechanical; a replayed headline automatically inherits its true date (98% of the
   June-23 batch is re-dated this way).
2. **Certification flag** — the date is *proven* to be the publication time iff the earliest message is
   `ADD_1STPASS`/`ADD_STORY` (publication by feed semantics). Headlines whose earliest message is
   `UPDATE_ATTRIBUTE` were published before the capture starts (or in a feed gap): keep them flagged, or
   drop them — a measured decision, not a guess.
3. **Acceptance tests** (all mechanical): share of ADD-first headlines ≈ 100% after the global pass ·
   numbered-update chains `(1),(2),(3)` time-ordered ·  the June-2013 spike disappears (demo below).

**Impact on the research pipeline:** any burst that feeds an emergence detector should survive
re-dating. Bulk re-tag days (27k–63k messages) are routine, so *every* spike in the keyword timelines
(0.27) and every detection window (0.24 / 0.28) is suspect until the corpus is rebuilt this way —
done in `scripts/preprocess_news.py` + notebook 0.26 (`news_corpus.parquet`).

In [6]:
# demo of the fix on the case study: re-date each 2013 hack* headline by its earliest capture 2011-2013
lookup = pl.concat([found[2011].select(["Headline", "CaptureTime"]),
                    found[2012].select(["Headline", "CaptureTime"])]
         ).sort("CaptureTime").unique(subset="Headline", keep="first").rename({"CaptureTime": "true_time"})
fixed = (h13.join(lookup, on="Headline", how="left")
            .with_columns(pl.coalesce(["true_time", "CaptureTime"]).alias("date_fixed")))
before = h13.group_by(pl.col("CaptureTime").dt.strftime("%Y-%m").alias("month")).len().rename({"len": "before"})
after = (fixed.filter(pl.col("date_fixed").dt.year() == 2013)
              .group_by(pl.col("date_fixed").dt.strftime("%Y-%m").alias("month")).len().rename({"len": "after"}))
cmp = before.join(after, on="month", how="full", coalesce=True).sort("month").fill_null(0)
print("monthly hack* headlines dated by CaptureTime (before) vs earliest-occurrence (after):")
print(cmp.to_pandas().to_string(index=False))
n_moved = fixed.filter(pl.col("true_time").is_not_null()).height
print(f"\n{n_moved:,} headlines re-dated out of 2013 -> the June spike collapses to background level")

monthly hack* headlines dated by CaptureTime (before) vs earliest-occurrence (after):
  month  before  after
2013-01      46     46
2013-02      83     83
2013-03     109    109
2013-04      62     62
2013-05      42     42
2013-06    1365    104
2013-07      66     66
2013-08      36     36
2013-09      34     34
2013-10      48     48
2013-11      89     89
2013-12      29     29

1,261 headlines re-dated out of 2013 -> the June spike collapses to background level


# FINAL DECISION:

1. la data della headline è quella del **primo evento ADD** (`ADD_1STPASS` *oppure* `ADD_STORY` — il 3.2% delle headline ha solo `ADD_STORY`)

2. una headline che entra solo come `UPDATE_ATTRIBUTE` viene **droppata** (~35% in-year; sono storie pre-capture o appartenenti all'anno del loro ADD — non una perdita di notizie del periodo)

Implementato in `scripts/preprocess_news.py` (rule `add-event-v1`) → `output/news_corpus.parquet`; verifica in 0.26.